# 06 — Data Preprocessing & Model Building

This notebook implements and verifies the tasks from **Epic 3: Data Pre-processing** and **Epic 4: Model Building**:
1. **Handling Outliers** using IQR winsorization (capping).
2. **Handling Categorical Values** using one-hot encoding.
3. **Splitting Data into Training and Test Sets** with stratified splitting.
4. **Feature Scaling** using StandardScaler.
5. **Decision Tree Model** training and hyperparameter tuning.
6. **Random Forest Model** training and hyperparameter tuning.

In [4]:
import sys
print(sys.executable)

c:\Users\Vijaykiran\Desktop\SkillWallet_Project\rising-waters-flood-prediction\env\Scripts\python.exe


In [5]:
import sys
import os
import pandas as pd
import numpy as np

# Add source code directory to path
sys.path.append(os.path.abspath(r"../../5. Project Development Phase/src"))

from preprocessing import load_data, handle_outliers, handle_missing_values, encode_categories, split_data, scale_features
from decision_tree import train_decision_tree, tune_hyperparameters as tune_dt
from random_forest import train_random_forest, tune_hyperparameters as tune_rf

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Load dataset
filepath = r"../../3. Project Design Phase/dataset/raw/flood dataset.xlsx"
df = load_data(filepath)
df = handle_missing_values(df)
print(f"Loaded dataset with shape: {df.shape}")
df.head()

Loaded dataset with shape: (115, 11)


,Temp,Humidity,Cloud Cover,ANNUAL,Jan-Feb,Mar-May,Jun-Sep,Oct-Dec,avgjune,sub,flood
0,29,70,30,3248.6,73.4,386.2,2122.8,666.1,274.866667,649.9,0
1,28,75,40,3326.6,9.3,275.7,2403.4,638.2,130.300000,256.4,1
2,28,75,42,3271.2,21.7,336.3,2343.0,570.1,186.200000,308.9,0
3,29,71,44,3129.7,26.7,339.4,2398.2,365.3,366.066667,862.5,0
4,31,74,40,2741.6,23.4,378.5,1881.5,458.1,283.400000,586.9,0


## 1. Handling Outliers

We use **IQR Winsorization (capping)** to handle outliers. Capping outliers preserves all rows (unlike dropping outliers), which is critical since our dataset is small (115 rows) and has a highly imbalanced target column (`flood` has only 16 positive examples).

In [6]:
# Inspect statistics before outlier capping
print("Max values BEFORE capping outliers:")
print(df.max())

# Apply outlier treatment
df_clean = handle_outliers(df)

print("\nMax values AFTER capping outliers:")
print(df_clean.max())

Max values BEFORE capping outliers:
Temp             31.000000
Humidity         79.000000
Cloud Cover      44.000000
ANNUAL         4257.800000
Jan-Feb          98.100000
Mar-May         915.200000
Jun-Sep        3451.300000
Oct-Dec         823.300000
avgjune         366.066667
sub             982.700000
flood             1.000000
dtype: float64

Max values AFTER capping outliers:
Temp             31.000000
Humidity         79.000000
Cloud Cover      44.000000
ANNUAL         3968.400000
Jan-Feb          88.625000
Mar-May         690.625000
Jun-Sep        2953.975000
Oct-Dec         823.300000
avgjune         366.066667
sub             982.700000
flood             1.000000
dtype: float64


## 2. Handling Categorical Values

Even though all columns in this dataset are currently numeric, we run `encode_categories` to ensure robust handling of non-numeric data types (e.g. converting categorical object columns to one-hot representation).

In [7]:
# Encode categories (if any)
df_encoded = encode_categories(df_clean)
print(f"Encoded dataset shape: {df_encoded.shape}")
print(df_encoded.dtypes)

Encoded dataset shape: (115, 11)
Temp             int64
Humidity         int64
Cloud Cover      int64
ANNUAL         float64
Jan-Feb        float64
Mar-May        float64
Jun-Sep        float64
Oct-Dec        float64
avgjune        float64
sub            float64
flood            int64
dtype: object


## 3. Splitting Data into Training and Test Sets

We separate features and target, and split into train and test sets using **stratified splitting** (`stratify=y`) to maintain the target class distribution in both splits.

In [8]:
X = df_encoded.drop(columns=['flood'])
y = df_encoded['flood']

X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]} samples")
print(f"Test size:  {X_test.shape[0]} samples")
print("\nTarget distribution in Training Set:")
print(y_train.value_counts(normalize=True))
print("\nTarget distribution in Test Set:")
print(y_test.value_counts(normalize=True))

Train size: 92 samples
Test size:  23 samples

Target distribution in Training Set:
flood
0    0.858696
1    0.141304
Name: proportion, dtype: float64

Target distribution in Test Set:
flood
0    0.869565
1    0.130435
Name: proportion, dtype: float64


## 4. Feature Scaling

We use `StandardScaler` to bring features onto the same scale (mean=0, variance=1) which is critical for distance-based and regularized algorithms.

In [9]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test, scaler_type="standard")
print("Training Set summary statistics after scaling:")
X_train_scaled.describe().loc[['mean', 'std', 'min', 'max']]

Training Set summary statistics after scaling:


,Temp,Humidity,Cloud Cover,ANNUAL,Jan-Feb,Mar-May,Jun-Sep,Oct-Dec,avgjune,sub
mean,-1.242967e-15,1.433636e-15,-1.957221e-16,1.212798e-15,-5.647656e-16,5.309762e-16,1.619779e-15,-3.529785e-16,-3.016910e-16,2.642814e-16
std,1.005479e+00,1.005479e+00,1.005479e+00,1.005479e+00,1.005479e+00,1.005479e+00,1.005479e+00,1.005479e+00,1.005479e+00,1.005479e+00
min,-1.477520e+00,-1.363258e+00,-1.426580e+00,-2.055924e+00,-1.265801e+00,-2.044195e+00,-1.563333e+00,-2.472931e+00,-2.308824e+00,-1.972310e+00
max,1.187810e+00,1.646815e+00,1.842877e+00,2.595465e+00,2.891200e+00,2.274650e+00,2.651562e+00,2.572493e+00,2.476262e+00,2.656716e+00


## 5. Decision Tree Model

We train a `DecisionTreeClassifier` and tune its hyper-parameters using grid search over depth, min samples split, and leaf nodes, using `f1` as the grid scoring metric.

In [10]:
dt_grid = {
    'max_depth': [3, 5, 8, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

best_dt, best_dt_params = tune_dt(X_train_scaled, y_train, dt_grid)
print(f"Best Decision Tree parameters found: {best_dt_params}")

# Predict & Evaluate
dt_preds = best_dt.predict(X_test_scaled)
print("\nDecision Tree Test Report:")
print(classification_report(y_test, dt_preds, zero_division=0))

Best Decision Tree parameters found: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}

Decision Tree Test Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98        20
           1       1.00      0.67      0.80         3

    accuracy                           0.96        23
   macro avg       0.98      0.83      0.89        23
weighted avg       0.96      0.96      0.95        23



## 6. Random Forest Model

We train a `RandomForestClassifier` with balanced class weights to penalize minority class errors and search over trees and maximum depth.

In [11]:
rf_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 8, None],
    'min_samples_split': [2, 5, 10]
}

best_rf, best_rf_params = tune_rf(X_train_scaled, y_train, rf_grid)
print(f"Best Random Forest parameters found: {best_rf_params}")

# Predict & Evaluate
rf_preds = best_rf.predict(X_test_scaled)
print("\nRandom Forest Test Report:")
print(classification_report(y_test, rf_preds, zero_division=0))

Best Random Forest parameters found: {'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 50}

Random Forest Test Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98        20
           1       1.00      0.67      0.80         3

    accuracy                           0.96        23
   macro avg       0.98      0.83      0.89        23
weighted avg       0.96      0.96      0.95        23



## Summary of Findings

### Data Analysis Key Findings
- The target variable is highly imbalanced with **16 flood cases** and **99 non-flood cases** out of 115 samples.
- Outliers capping successfully adjusted the maximum rainfall and temperature ranges without losing valuable flood rows.
- Scaling has centered features around 0, making model weights comparable.
- GridSearchCV hyperparameter tuning has optimized Decision Tree and Random Forest parameters based on minority class F1-Score.

### Insights or Next Steps
- Run all cells in this notebook to see output statistics and compare model performance metrics on your local environment.